# RIDGE REGRESSION (L2)

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
data_df = pd.read_csv("data_cleaned.csv")
print(f"Loaded {len(data_df)} rows, {len(data_df.columns)} columns")
print(f"Columns: {list(data_df.columns)}")
data_df.head()

Loaded 1328 rows, 9 columns
Columns: ['age', 'bmi', 'children', 'charges', 'gender_encoded', 'smoker_encoded', 'region_northwest', 'region_southeast', 'region_southwest']


,age,bmi,children,charges,gender_encoded,smoker_encoded,region_northwest,region_southeast,region_southwest
0,19,27.900,0,16884.92400,1,1,0.0,0.0,1.0
1,18,33.770,1,1725.55230,0,0,0.0,1.0,0.0
2,28,33.000,3,4449.46200,0,0,0.0,1.0,0.0
3,33,22.705,0,21984.47061,0,0,1.0,0.0,0.0
4,32,28.880,0,3866.85520,0,0,1.0,0.0,0.0


## 1. Data Splitting

In [23]:
from sklearn.model_selection import train_test_split

data_df["smoker_bmi"] = data_df["smoker_encoded"] * data_df["bmi"]
data_df["age_squared"] = data_df["age"] ** 2

feature_cols = [
    "age", "age_squared", "bmi", "children", "gender_encoded", "smoker_encoded",
    "region_northwest", "region_southeast", "region_southwest", "smoker_bmi"
]

X = data_df[feature_cols]
y = data_df["charges"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("Features:", feature_cols)

X_train: (1062, 10)
X_test : (266, 10)
Features: ['age', 'age_squared', 'bmi', 'children', 'gender_encoded', 'smoker_encoded', 'region_northwest', 'region_southeast', 'region_southwest', 'smoker_bmi']


## 2. Feature Scaling (RobustScaler)

In [24]:
from sklearn.preprocessing import RobustScaler

# Feature scaling
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Robust scaling completed.")
print("X_train_scaled:", X_train_scaled.shape)
print("X_test_scaled :", X_test_scaled.shape)

Robust scaling completed.
X_train_scaled: (1062, 10)
X_test_scaled : (266, 10)


## 3. Tune Alpha + Train Final Model + Evaluation

In [25]:
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

alpha_list = [0.01, 0.1, 1, 10, 100, 500, 1000]
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# 1) Find best alpha on train set only
ridge_cv = RidgeCV(alphas=alpha_list, cv=cv, scoring="neg_root_mean_squared_error")
ridge_cv.fit(X_train_scaled, y_train)
best_alpha = float(ridge_cv.alpha_)

# 2) Train final model once with best alpha
ridge_final = Ridge(alpha=best_alpha)
ridge_final.fit(X_train_scaled, y_train)

# Keep variable name for downstream cells (coefficient section)
ridge_model = ridge_final

y_pred_train = ridge_model.predict(X_train_scaled)
y_pred = ridge_model.predict(X_test_scaled)

# Final metrics (tuned model)
mae_train = mean_absolute_error(y_train, y_pred_train)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
r2_train = r2_score(y_train, y_pred_train)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

# Optional: compact baseline vs tuned comparison (train + test)
ridge_baseline = Ridge(alpha=1.0)
ridge_baseline.fit(X_train_scaled, y_train)

y_pred_baseline_train = ridge_baseline.predict(X_train_scaled)
y_pred_baseline_test = ridge_baseline.predict(X_test_scaled)

comparison = pd.DataFrame([
    {
        "model": "Ridge baseline (alpha=1.0)",
        "R2_train": r2_score(y_train, y_pred_baseline_train),
        "RMSE_train": np.sqrt(mean_squared_error(y_train, y_pred_baseline_train)),
        "MAE_train": mean_absolute_error(y_train, y_pred_baseline_train),
        "R2_test": r2_score(y_test, y_pred_baseline_test),
        "RMSE_test": np.sqrt(mean_squared_error(y_test, y_pred_baseline_test)),
        "MAE_test": mean_absolute_error(y_test, y_pred_baseline_test),
    },
    {
        "model": f"Ridge tuned (alpha={best_alpha:g})",
        "R2_train": r2_train,
        "RMSE_train": rmse_train,
        "MAE_train": mae_train,
        "R2_test": r2,
        "RMSE_test": rmse,
        "MAE_test": mae,
    },
])

print(f"Best alpha from CV: {best_alpha}")
print("=== Final Ridge Evaluation (Tuned) ===")
print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²  : {r2:.4f}")

comparison

Best alpha from CV: 0.01
=== Final Ridge Evaluation (Tuned) ===
MAE : 2796.22
RMSE: 4705.77
R²  : 0.8407


,model,R2_train,RMSE_train,MAE_train,R2_test,RMSE_test,MAE_test
0,Ridge baseline (alpha=1.0),0.840729,4806.333213,2876.527738,0.839828,4718.447733,2778.736620
1,Ridge tuned (alpha=0.01),0.841092,4800.839935,2880.422732,0.840687,4705.774087,2796.221883
